# Module 2 - Build the Leakage-Resistant BANKING77 Manifest

**Audience:** engineers preparing auditable intent-classification data in VS Code.

**Prerequisites:** Module 1, the project virtual environment and internet access for the first run.

**Learning goals:** acquire commit-pinned BANKING77 files, construct a group-stratified validation set, quarantine exact leakage, validate the untouched-test contract and inspect a text-free manifest.


## Outline

1. Load the versioned dataset configuration.
2. Download or reuse the commit-pinned source files.
3. Build train and validation splits by normalised-text group.
4. inspect hashes, counts, quarantine reasons and overlap checks.
5. Confirm that the manifest contains no message text.


In [10]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    path
    for path in [current, *current.parents]
    if (path / 'pyproject.toml').exists()
)
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT.name


'governed-banking-intent-router'

## Step 1 - Inspect the immutable source policy

The configuration uses a full Git commit rather than a mutable branch. The official test file is acquired with the training file but is not used to choose the validation split.


In [11]:
from governed_banking.data import DatasetConfig, acquire_and_prepare, validate_manifest

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'dataset.yaml'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'banking77'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'manifests' / 'banking77-seed-42.json'
config = DatasetConfig.from_yaml(CONFIG_PATH)
{
    'source_commit': config.commit,
    'expected_train_rows': config.expected_train_rows,
    'expected_test_rows': config.expected_test_rows,
    'expected_labels': config.expected_label_count,
    'seed': config.seed,
    'validation_fraction': config.validation_fraction,
}


{'source_commit': '57ec275d8078af65b7731c2a98be812d844a6d6b',
 'expected_train_rows': 10003,
 'expected_test_rows': 3080,
 'expected_labels': 77,
 'seed': 42,
 'validation_fraction': 0.15}

## Step 2 - Acquire and prepare

The first run downloads three small files. Later runs automatically use offline mode when all files are present. Raw files are ignored by Git; the manifest is safe to commit because it contains indices and hashes rather than message text.


In [12]:
required_files = [RAW_DIR / filename for filename in config.files.values()]
offline = all(path.exists() for path in required_files)
manifest = acquire_and_prepare(
    CONFIG_PATH,
    RAW_DIR,
    MANIFEST_PATH,
    offline=offline,
)
validate_manifest(manifest)
print(f'Validated manifest: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}')
print(f'Content hash: {manifest["manifest_sha256"]}')


Validated manifest: data/manifests/banking77-seed-42.json
Content hash: ba4499586c47a941d70f300b41c3a0b0e624d5405cc1ec2079d6779a4577e5b2


## Step 3 - Inspect split counts and source hashes

Quarantined rows are excluded only from the development pool. The official test indices must remain the complete ordered sequence from 0 to 3,079.


In [13]:
split_summary = pd.DataFrame(
    [
        {
            'split': name,
            'count': details['count'],
            'index_hash': details['source_indices_sha256'][:12],
        }
        for name, details in manifest['splits'].items()
    ]
)
assert manifest['splits']['test']['source_indices'] == list(range(3_080))
split_summary


,split,count,index_hash
0,train,8495,5a4325b55dc9
1,validation,1501,9c66b4850a18
2,test,3080,4b01cf82559f


In [14]:
pd.DataFrame(
    [
        {
            'source': name,
            'sha256': details['sha256'],
            'bytes': details['bytes'],
        }
        for name, details in manifest['sources'].items()
    ]
)


,source,sha256,bytes
0,categories,53261da888122daf2d120d925458631d9619e15d82e560...,2036
1,official_train,b06e26ac675513959a63135f11b94ea7786ed02da65db9...,839073
2,official_test,d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a...,239961


## Step 4 - Inspect leakage and quarantine decisions

All final cross-split overlap counts must be zero. The original overlap counters are evidence about the source data and may be non-zero.


In [15]:
quarantine_reasons = Counter(row['reason'] for row in manifest['quarantined_train'])
{
    'integrity': manifest['integrity'],
    'quarantined_train_rows': len(manifest['quarantined_train']),
    'quarantine_reasons': dict(quarantine_reasons),
}


{'integrity': {'original_train_test_overlap_groups': 7,
  'original_conflicting_training_groups': 0,
  'final_train_validation_overlap_groups': 0,
  'final_train_test_overlap_groups': 0,
  'final_validation_test_overlap_groups': 0,
  'train_duplicate_groups': 2,
  'validation_duplicate_groups': 2,
  'test_duplicate_groups': 1},
 'quarantined_train_rows': 7,
 'quarantine_reasons': {'duplicates_official_test': 7}}

## Step 5 - Confirm manifest minimisation

The manifest should contain provenance, hashes, labels and integer indices, but no `text` field or message content.


In [16]:
encoded_manifest = json.dumps(manifest)
assert manifest['contains_message_text'] is False
assert '"text"' not in encoded_manifest
print('Manifest minimisation check passed.')


Manifest minimisation check passed.


## Exercise

For each split, calculate the smallest and largest class counts. Explain why macro-F1 will be reported even if the label distribution appears relatively balanced.


In [17]:
class_balance = {}
for split_name, details in manifest['splits'].items():
    counts = list(details['label_distribution'].values())
    class_balance[split_name] = {'minimum': min(counts), 'maximum': max(counts)}
class_balance


{'train': {'minimum': 30, 'maximum': 159},
 'validation': {'minimum': 5, 'maximum': 28},
 'test': {'minimum': 40, 'maximum': 40}}

## Pitfalls and next step

- Do not select checkpoints or thresholds using the official test split.
- Do not split rows before grouping duplicate normalised messages.
- Do not silently delete source anomalies; quarantine and count them.
- Do not commit the raw CSV files. Commit the text-free manifest.
- Changing the seed, validation fraction or normalisation creates a new experiment version.

After the manifest is reviewed and committed, Module 3 will implement the TF-IDF logistic-regression baseline.
